In [3]:
"""
Ex 4.1 — Data fitting (scattered data interpolation) in NumPy

Implements and evaluates:
  (A) RBF interpolation / regression with a Gaussian kernel (with ridge regularization)
  (B) Inverse-distance weighting (IDW) baseline

Includes:
  - Random smoothly varying 2D function (sum of Gaussians)
  - Random sampling of a few dozen points
  - Fitting + error measurement on a regular grid
  - Parameter tuning via K-fold CV over (sigma, lambda)
  - Repeat experiment and report how optimal parameters change
  - Optional: piecewise-smooth function (harder) + quick mitigation ideas

Dependencies: numpy, matplotlib
Run: python ex4_1_data_fitting.py
"""

import numpy as np
import matplotlib.pyplot as plt


# -----------------------------
# 1) Generate a random 2D function
# -----------------------------
def make_random_gaussian_field(seed=0, K=6, domain=(0.0, 1.0)):
    """
    Returns a callable f(XY) -> values, where XY is (..., 2).
    Field is a sum of K Gaussians with random centers, amplitudes, and scales.
    """
    rng = np.random.default_rng(seed)
    lo, hi = domain

    centers = rng.uniform(lo, hi, size=(K, 2))
    amps = rng.normal(loc=0.0, scale=1.0, size=(K,))
    # scales in coordinate units; smaller = higher frequency/detail
    scales = rng.uniform(0.05, 0.25, size=(K,))

    def f(XY):
        # XY: (..., 2)
        diff = XY[..., None, :] - centers[None, None, :, :] if XY.ndim == 2 else XY[..., None, :] - centers
        # squared distance to each center
        r2 = np.sum(diff * diff, axis=-1)
        # Gaussian bumps
        g = np.exp(-0.5 * r2 / (scales * scales))
        return np.sum(g * amps, axis=-1)

    return f


def make_piecewise_field(seed=0, split_x=0.5, K_left=5, K_right=5):
    """
    Piecewise-smooth field: different Gaussian mixtures on left/right halves.
    Harder than globally smooth.
    """
    fL = make_random_gaussian_field(seed=seed, K=K_left)
    fR = make_random_gaussian_field(seed=seed + 1, K=K_right)

    def f(XY):
        x = XY[..., 0]
        out = np.empty_like(x, dtype=float)
        mask = x < split_x
        out[mask] = fL(XY[mask])
        out[~mask] = fR(XY[~mask])
        return out

    return f


# -----------------------------
# 2) Sampling utilities
# -----------------------------
def sample_function(f, n_samples=60, seed=0, noise_std=0.0, domain=(0.0, 1.0)):
    rng = np.random.default_rng(seed)
    lo, hi = domain
    X = rng.uniform(lo, hi, size=(n_samples, 2))
    y = f(X)
    if noise_std > 0:
        y = y + rng.normal(0.0, noise_std, size=y.shape)
    return X, y


def make_grid(n=100, domain=(0.0, 1.0)):
    lo, hi = domain
    xs = np.linspace(lo, hi, n)
    ys = np.linspace(lo, hi, n)
    XX, YY = np.meshgrid(xs, ys, indexing="xy")
    grid_xy = np.stack([XX.ravel(), YY.ravel()], axis=1)
    return XX, YY, grid_xy


# -----------------------------
# 3) Scattered data interpolation techniques
# -----------------------------
def pairwise_sq_dists(A, B):
    """
    Compute squared Euclidean distances between points in A (N,2) and B (M,2).
    Returns (N,M).
    Pure NumPy, array-ops only.
    """
    # (N,1,2) - (1,M,2) -> (N,M,2)
    diff = A[:, None, :] - B[None, :, :]
    return np.sum(diff * diff, axis=2)


def rbf_fit(X, y, sigma, lam):
    """
    Fit weights w for Gaussian RBF:
      f(x) = sum_i w_i * exp(-||x-x_i||^2 / (2*sigma^2))

    With ridge regularization:
      minimize ||K w - y||^2 + lam ||w||^2
      => (K^T K + lam I) w = K^T y
    Here K is symmetric, so K^T K = K^2, but we use the standard stable form.

    Returns: weights w (N,)
    """
    r2 = pairwise_sq_dists(X, X)
    K = np.exp(-0.5 * r2 / (sigma * sigma))

    # Solve (K + lam I) w = y is also common (kernel ridge with centered features).
    # We'll use (K + lam I) directly for simplicity and conditioning.
    A = K + lam * np.eye(K.shape[0])
    w = np.linalg.solve(A, y)
    return w


def rbf_predict(X_train, w, X_query, sigma):
    r2 = pairwise_sq_dists(X_query, X_train)
    Kq = np.exp(-0.5 * r2 / (sigma * sigma))
    return Kq @ w


def idw_predict(X_train, y_train, X_query, power=2.0, eps=1e-12):
    """
    Inverse Distance Weighting:
      y(x) = sum_i (y_i / d_i^p) / sum_i (1 / d_i^p)
    """
    y_train = np.asarray(y_train).reshape(-1)  # <-- force (n_train,)
    diff = X_query[:, None, :] - X_train[None, :, :]
    d2 = np.sum(diff * diff, axis=2) + eps
    w = 1.0 / (d2 ** (power / 2.0))            # (n_query, n_train)
    w_sum = np.sum(w, axis=1)                 # (n_query,)
    return (w @ y_train) / w_sum              # (n_query,)



# -----------------------------
# 4) Error measurement + visualization
# -----------------------------
def mse(a, b):
    e = a - b
    return float(np.mean(e * e))


def plot_results(XX, YY, truth, pred, title):
    Zt = truth.reshape(XX.shape)
    Zp = pred.reshape(XX.shape)
    Ze = (Zp - Zt)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].set_title("Ground truth")
    im0 = axes[0].imshow(Zt, origin="lower", extent=[XX.min(), XX.max(), YY.min(), YY.max()])
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    axes[1].set_title(title)
    im1 = axes[1].imshow(Zp, origin="lower", extent=[XX.min(), XX.max(), YY.min(), YY.max()])
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    axes[2].set_title("Error (pred - truth)")
    im2 = axes[2].imshow(Ze, origin="lower", extent=[XX.min(), XX.max(), YY.min(), YY.max()])
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    for ax in axes:
        ax.set_xlabel("x")
        ax.set_ylabel("y")
    fig.tight_layout()
    plt.show()


# -----------------------------
# 5) Cross-validation for parameter tuning
# -----------------------------
def kfold_indices(n, k, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)
    return folds


def cv_rbf(X, y, sigmas, lams, k=5, seed=0):
    """
    Grid search over sigma, lambda with K-fold CV.
    Returns best (sigma, lam) and a results array of shape (len(sigmas), len(lams)).
    """
    folds = kfold_indices(len(X), k, seed=seed)
    scores = np.zeros((len(sigmas), len(lams)), dtype=float)

    for si, sigma in enumerate(sigmas):
        for li, lam in enumerate(lams):
            fold_mse = []
            for fi in range(k):
                val_idx = folds[fi]
                tr_idx = np.concatenate([folds[j] for j in range(k) if j != fi])

                Xtr, ytr = X[tr_idx], y[tr_idx]
                Xva, yva = X[val_idx], y[val_idx]

                w = rbf_fit(Xtr, ytr, sigma=sigma, lam=lam)
                yhat = rbf_predict(Xtr, w, Xva, sigma=sigma)
                fold_mse.append(mse(yhat, yva))
            scores[si, li] = np.mean(fold_mse)

    best_flat = int(np.argmin(scores))
    best_si, best_li = np.unravel_index(best_flat, scores.shape)
    return (sigmas[best_si], lams[best_li]), scores


# -----------------------------
# 6) Full experiment runner
# -----------------------------
def run_experiment(
    seed_func=0,
    seed_samples=1,
    n_samples=60,
    grid_n=120,
    noise_std=0.0,
    piecewise=False,
    show_plots=True,
):
    if piecewise:
        f = make_piecewise_field(seed=seed_func)
        name = "Piecewise-smooth field"
    else:
        f = make_random_gaussian_field(seed=seed_func)
        name = "Smooth field"

    Xs, ys = sample_function(f, n_samples=n_samples, seed=seed_samples, noise_std=noise_std)
    XX, YY, grid_xy = make_grid(n=grid_n)
    y_true = f(grid_xy)

    # Baseline: IDW
    y_idw = idw_predict(Xs, ys, grid_xy, power=2.0)
    idw_err = mse(y_idw, y_true)

    # RBF + CV
    sigmas = np.logspace(np.log10(0.02), np.log10(0.4), 14)
    lams = np.logspace(-10, -2, 9)  # small to moderate ridge
    (best_sigma, best_lam), cv_scores = cv_rbf(Xs, ys, sigmas, lams, k=5, seed=seed_samples)

    w = rbf_fit(Xs, ys, sigma=best_sigma, lam=best_lam)
    y_rbf = rbf_predict(Xs, w, grid_xy, sigma=best_sigma)
    rbf_err = mse(y_rbf, y_true)

    print("========================================")
    print(f"{name}")
    print(f"Samples: {n_samples}, noise_std={noise_std}")
    print(f"IDW MSE on grid:  {idw_err:.6g}")
    print(f"RBF best sigma:   {best_sigma:.6g}")
    print(f"RBF best lambda:  {best_lam:.6g}")
    print(f"RBF MSE on grid:  {rbf_err:.6g}")

    if show_plots:
        # show sample locations overlaid on truth for debugging
        plt.figure(figsize=(5, 4))
        plt.title(f"{name}: sample locations")
        Zt = y_true.reshape(XX.shape)
        plt.imshow(Zt, origin="lower", extent=[XX.min(), XX.max(), YY.min(), YY.max()])
        plt.scatter(Xs[:, 0], Xs[:, 1], s=18)
        plt.xlabel("x")
        plt.ylabel("y")
        plt.colorbar(fraction=0.046)
        plt.tight_layout()
        plt.show()

        plot_results(XX, YY, y_true, y_idw, title="IDW prediction")
        plot_results(XX, YY, y_true, y_rbf, title=f"RBF prediction (sigma={best_sigma:.3g}, lam={best_lam:.1e})")

        # visualize CV surface
        plt.figure(figsize=(6, 4))
        plt.title("CV MSE (lower is better)")
        plt.imshow(
            cv_scores,
            origin="lower",
            aspect="auto",
            extent=[np.log10(lams[0]), np.log10(lams[-1]), np.log10(sigmas[0]), np.log10(sigmas[-1])],
        )
        plt.xlabel("log10(lambda)")
        plt.ylabel("log10(sigma)")
        plt.colorbar(fraction=0.046)
        plt.tight_layout()
        plt.show()

    return {
        "best_sigma": best_sigma,
        "best_lam": best_lam,
        "idw_mse": idw_err,
        "rbf_mse": rbf_err,
    }


# -----------------------------
# Main: run twice and compare best params
# -----------------------------
if __name__ == "__main__":
    # First run (smooth)
    r1 = run_experiment(
        seed_func=0,
        seed_samples=1,
        n_samples=60,
        grid_n=120,
        noise_std=0.00,
        piecewise=False,
        show_plots=True,
    )

    # 6) Repeat with new random sample locations (same underlying function)
    r2 = run_experiment(
        seed_func=0,
        seed_samples=2,
        n_samples=60,
        grid_n=120,
        noise_std=0.00,
        piecewise=False,
        show_plots=False,
    )

    print("\nParameter change across repeats (same f, new samples):")
    print(f"  sigma:  {r1['best_sigma']:.6g} -> {r2['best_sigma']:.6g}")
    print(f"  lambda: {r1['best_lam']:.6g} -> {r2['best_lam']:.6g}")

    # 7) Optional: piecewise-smooth test function (harder)
    rp = run_experiment(
        seed_func=10,
        seed_samples=3,
        n_samples=70,
        grid_n=120,
        noise_std=0.00,
        piecewise=True,
        show_plots=True,
    )

    print("\nNotes on piecewise-smooth case:")
    print("  Expect higher error because one global smoothness scale (sigma) struggles across a discontinuity-like region.")
    print("  Mitigations to try:")
    print("    - Locally adaptive bandwidth: sigma(x) based on kNN distances or local density.")
    print("    - Local RBF / moving least squares: fit using only k nearest samples per query point.")
    print("    - Partition-of-unity / tiling: fit separate models in overlapping regions and blend.")
    print("    - Add an edge-aware weight (like bilateral) if you have an image/guide signal.")


IndexError: index 23 is out of bounds for axis 0 with size 1